# AnimeGANv3 — Video to Animation on Colab T4
Select a **T4 GPU**, then **Runtime → Run all**. The notebook clones the benchmark branch, installs dependencies, downloads the Hayao model automatically, asks for one video, renders it, and reports throughput.

In [ ]:
import os, subprocess, sys
REPO = '/content/video-to-animation'
if not os.path.exists(REPO):
    subprocess.run(['git','clone','-b','animeganv3-baseline','https://github.com/v-tech-hub/video-to-animation.git',REPO],check=True)
os.chdir(REPO)
print('Working directory:', os.getcwd())
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip uninstall -y -q onnxruntime onnxruntime-gpu || true
!pip install -q -r requirements-colab.txt


In [ ]:
import onnxruntime as ort, subprocess
print('Python:', sys.version)
print('ORT version:', ort.__version__)
print('ORT device:', ort.get_device())
print('Available providers:', ort.get_available_providers())
print('GPU snapshot:')
subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu','--format=csv,noheader'])
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDAExecutionProvider unavailable.'
# Smoke-test an actual CUDA session now; provider presence alone is not enough.
MODEL_SMOKE_PENDING = True


In [ ]:
from pathlib import Path
,import urllib.request
,MODEL = '/content/AnimeGANv3_Hayao_36.onnx'
,MODEL_URL = 'https://github.com/TachibanaYoshino/AnimeGANv3/releases/download/v1.1.0/AnimeGANv3_Hayao_36.onnx'
,if not Path(MODEL).exists():
,    print('Downloading AnimeGANv3 Hayao model...')
,    urllib.request.urlretrieve(MODEL_URL, MODEL)
,print('Model:', MODEL, f'({Path(MODEL).stat().st_size / 1024 / 1024:.2f} MB)')
print('Creating CUDA smoke-test session...')
,smoke = ort.InferenceSession(MODEL, providers=['CUDAExecutionProvider'])
,print('Smoke-test active providers:', smoke.get_providers())
,assert smoke.get_providers()[0] == 'CUDAExecutionProvider', 'CUDA provider failed to activate; stop before uploading/rendering.'
,del smoke
,print('CUDA smoke test: OK')


## Upload your video
Upload **one video only**. The notebook copies it to a simple filename so spaces or `(1)` in the original filename cannot cause trouble.

In [ ]:
from google.colab import files
import shutil
uploaded = files.upload()
videos = [n for n in uploaded if Path(n).suffix.lower() in {'.mp4','.mov','.avi','.mkv','.webm'}]
assert videos, 'Upload a video file.'
src = Path(videos[0])
VIDEO = '/content/input' + src.suffix.lower()
shutil.copyfile(src, VIDEO)
print('Video:', VIDEO)


In [ ]:
import time, sys, threading
os.chdir(REPO)
os.makedirs('output',exist_ok=True)
cmd=['python','-u',os.path.join(REPO,'tools','video2anime.py'),'-i',VIDEO,'-o',os.path.join(REPO,'output'),'-m',MODEL,'-d','gpu']
print('=== AnimeGANv3 render ===', flush=True)
print('Working directory:', os.getcwd(), flush=True)
print('Model:', MODEL, flush=True)
print('Input:', VIDEO, flush=True)
print('Command:', ' '.join(cmd), flush=True)
print('\n--- live process output ---', flush=True)
stop_gpu = threading.Event()
def gpu_monitor():
    while not stop_gpu.wait(15):
        q = subprocess.run(['nvidia-smi','--query-gpu=utilization.gpu,memory.used,power.draw','--format=csv,noheader,nounits'],capture_output=True,text=True)
        if q.returncode == 0:
            print('[gpu] util%, memMiB, powerW:', q.stdout.strip(), flush=True)
threading.Thread(target=gpu_monitor, daemon=True).start()
start=time.perf_counter()
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout:
    print(line,end='',flush=True)
returncode=proc.wait()
stop_gpu.set()
elapsed=time.perf_counter()-start
print('\n--- process finished ---', flush=True)
print(f'Exit code: {returncode}', flush=True)
print(f'Wall time: {elapsed:.2f} s', flush=True)
if returncode != 0:
    raise RuntimeError(f'AnimeGANv3 failed with exit code {returncode}')


In [ ]:
import time
os.chdir(REPO)
os.makedirs('output',exist_ok=True)
cmd=['python',os.path.join(REPO,'tools','video2anime.py'),'-i',VIDEO,'-o',os.path.join(REPO,'output'),'-m',MODEL,'-d','gpu']
print('Running:', ' '.join(cmd))
start=time.perf_counter()
result=subprocess.run(cmd,text=True,capture_output=True)
elapsed=time.perf_counter()-start
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n',result.stderr)
    raise RuntimeError(f'AnimeGANv3 failed with exit code {result.returncode}')
print(f'Wall time: {elapsed:.2f} s')


In [ ]:
import cv2
cap=cv2.VideoCapture(VIDEO)
frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps=cap.get(cv2.CAP_PROP_FPS); cap.release()
duration=frames/fps if fps else 0
render_fps=frames/elapsed if elapsed else 0
rtf=elapsed/duration if duration else 0
print(f'Frames: {frames}')
print(f'Source FPS: {fps:.3f}')
print(f'Source duration: {duration:.2f} s')
print(f'Render throughput: {render_fps:.2f} FPS')
print(f'Realtime factor: {rtf:.2f}x')
outs=sorted(Path(REPO,'output').glob('*'),key=lambda x:x.stat().st_mtime,reverse=True)
print('Latest output:',outs[0] if outs else 'not found')
